In [1]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge

Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 332, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 332 (delta 81), reused 49 (delta 22), pack-reused 202 (from 1)
Receiving objects: 100% (332/332), 1.15 MiB | 11.53 MiB/s, done.
Resolving deltas: 100% (208/208), done.
/content/InkubaLM-Challenge


In [2]:
!git checkout refactor-src-structure
!git pull

Branch 'refactor-src-structure' set up to track remote branch 'refactor-src-structure' from 'origin'.
Switched to a new branch 'refactor-src-structure'
Already up to date.


In [3]:
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is 

In [4]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate peft datasets bitsandbytes trl

# Cell 2: Imports and setup
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset, Dataset
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, Gemma3ForConditionalGeneration

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [5]:
import sys
sys.path.append("..")  # Add parent directory to the path

import os
from typing import List
from pathlib import Path
import numpy as np

# DO NOT EDIT
# create submission file
import pandas as pd
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
)

from utils import (
    model_function,
    eval
    )

from src import (
    data_utils,
    model_utils,
    inference,
    prompts,
    evaluation,
    data_augment
    )

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split

In [6]:
#os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub import login

try:
    from google.colab import userdata

    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    # userdata.get("HF_TOKEN") indicates that the name of the token in the Colab env is HF_TOKEN
    os.environ["hf_token_2"] = userdata.get("hf_token_2")
except:
    os.environ["hf_token_2"] = "----"

login(token=os.environ["hf_token_2"])

token = os.environ["hf_token_2"]
if token == "----":
    print("⚠️ Warning: No Hugging Face token found. Some models may not load.")
else:
    login(token=token)

In [7]:
hf_token_2 = '...' # paste your token here
os.environ["HF_TOKEN"] = hf_token_2


In [ ]:
from huggingface_hub import login
login(token=hf_token_2)  # Force login using this token


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [36]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("lelapa/InkubaLM-0.4B", token=hf_token_2)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # or add a new token
model = AutoModelForCausalLM.from_pretrained("lelapa/InkubaLM-0.4B", token=hf_token_2)


VulavulaLlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [49]:
from peft import prepare_model_for_kbit_training
model_id = "lelapa/InkubaLM-0.4B"
model = AutoModelForCausalLM.from_pretrained(model_id,load_in_4bit=True,
    device_map="auto",trust_remote_code=True, token=hf_token_2)
model = prepare_model_for_kbit_training(model)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


In [50]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # <- Check this matches Inkuba arch
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)


In [28]:
import json
file_path = '/content/'

all_tasks = []
for fname in [os.path.join(file_path,"gemma_sentiment_soft_labels.jsonl"), os.path.join(file_path, "gemma_xnli_soft_labels_mc.jsonl"),  os.path.join(file_path, "gemma_mt_distilled.jsonl")]:
    with open(fname) as f:
        all_tasks.extend(json.loads(line) for line in f)

with open(os.path.join(file_path, "multitask_distill.jsonl"), 'w') as f:
    for row in all_tasks:
        f.write(json.dumps(row) + "\n")


In [29]:
import os
import json

file_path = '/content/'

# Infer language from ID or langs field
def infer_lang(example):
    if example["task"] == "translation":
        return example.get("langs", "eng-swa")
    elif "swa" in example["ID"]:
        return "swahili"
    elif "hau" in example["ID"]:
        return "hausa"
    else:
        return "unknown"

# Build prompt without <lang> tokens
def format_prompt(ex):
    instruction = ex.get("instruction", "").strip()
    input_text = ex.get("input", "").strip()
    return f"{instruction}\n{input_text}".strip() if input_text else instruction

# Load and transform data
input_files = [
    os.path.join(file_path, "gemma_sentiment_soft_labels.jsonl"),
    os.path.join(file_path, "gemma_xnli_soft_labels_mc.jsonl"),
    os.path.join(file_path, "gemma_mt_distilled.jsonl"),
]

all_tasks = []
for fname in input_files:
    with open(fname) as f:
        for line in f:
            ex = json.loads(line)
            ex["lang"] = infer_lang(ex)
            ex["prompt"] = format_prompt(ex)
            all_tasks.append(ex)

# Save to final multitask JSONL
output_path = os.path.join(file_path, "multitask_distill.jsonl")
with open(output_path, "w") as f:
    for row in all_tasks:
        f.write(json.dumps(row) + "\n")

print(f"✅ Saved multitask distillation dataset to: {output_path}")


✅ Saved multitask distillation dataset to: /content/multitask_distill.jsonl


In [32]:
from datasets import Dataset

dataset = Dataset.from_list(all_tasks)


In [121]:
dataset[0:10]

{'ID': ['ID_f3c74c7b_sentiment_test__hausa',
  'ID_aad19dbf_sentiment_test__hausa',
  'ID_f6de0381_sentiment_test__hausa',
  'ID_cbec84fe_sentiment_test__swahili',
  'ID_885caf5c_sentiment_test__hausa',
  'ID_7b906b9a_sentiment_test__hausa',
  'ID_c1b27b2b_sentiment_test__hausa',
  'ID_a5e55896_sentiment_test__swahili',
  'ID_4ba46a40_sentiment_test__swahili',
  'ID_d8cf7774_sentiment_test__hausa'],
 'task': ['sentiment',
  'sentiment',
  'sentiment',
  'sentiment',
  'sentiment',
  'sentiment',
  'sentiment',
  'sentiment',
  'sentiment',
  'sentiment'],
 'instruction': ["Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.",
  "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.",
  "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.",
  "Your task is

In [100]:
def tokenize_example(example):
    enc = tokenizer(
        example["prompt"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    dec = tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": dec["input_ids"],
        "soft_label": example.get("soft_label", None),
        "task": example["task"]
    }



In [101]:
from datasets import Dataset

dataset = Dataset.from_list(all_tasks)
tokenized_dataset = dataset.map(tokenize_example)

# Keep task + soft_label as is
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
    output_all_columns=True
)


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [102]:
tokenized_dataset

Dataset({
    features: ['ID', 'task', 'instruction', 'input', 'output', 'soft_label', 'lang', 'prompt', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 900
})

In [124]:
model, tokenizer, _ = setup_model_and_tokenizer_for_distillation("lelapa/InkubaLM-0.4B")
model = apply_lora_adapters(model)


trainable params: 524,288 || all params: 422,463,488 || trainable%: 0.1241


In [132]:
dataset

Dataset({
    features: ['ID', 'task', 'instruction', 'input', 'output', 'soft_label', 'lang', 'prompt'],
    num_rows: 900
})

In [147]:
dataset = Dataset.from_list(all_tasks)

In [161]:
def formatting_prompts_func(example):
    """
    Format distilled Gemma examples for instruction tuning.
    Assumes prompt already includes instruction + input merged.

    Args:
        example (dict): Contains prompt, output, and optional soft_label

    Returns:
        str: Formatted prompt for SFTTrainer
    """
    # Use prompt as instruction, and keep input blank
    instruction = example["prompt"]
    output = example["output"]

    # This formatting matches the response template expected by TRL's collator
    return f"### Instruction: {instruction}\n### Input:\n### Response: {output}"


In [164]:
test = dataset[0]
test_output = formatting_prompts_func(test)

In [165]:
test_output

"### Instruction: Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.\n@user ynxu fha da kanada kudi shikenan duk kayan nan zasu iya zama naka no🧢\n### Input:\n### Response: negative"

In [222]:
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import os
import re



# -------- 1. Load model & tokenizer --------
def setup_model_and_tokenizer_for_distillation(model_name, use_4bit=True):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=use_4bit,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    hf_token = os.environ.get("HF_TOKEN", None)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token if hf_token and hf_token != "----" else None,
    )

    return model, tokenizer, bnb_config


# -------- 2. Apply LoRA adapters --------
def apply_lora_adapters(model, r=8, lora_alpha=16, dropout=0.05):
    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=dropout,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model



def clean_text(text):
    return re.sub(r'[^\x00-\x7F]+', '', text)

def formatting_prompts_func(example):
    prompt = clean_text(example["prompt"]).strip()
    response = clean_text(example["output"]).strip()
    return f"{prompt}\n### Response: {response}"


# -------- 4. Setup trainer --------
def setup_distillation_trainer(
    model,
    dataset,
    tokenizer,
    output_dir,
    val_dataset=None,
    num_epochs=3,
    eval_steps=10
):
    # MATCH this with what your formatting function outputs!
    response_template = "\n### Response:"

    data_collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template,
        tokenizer=tokenizer,
        mlm=False
    )

    train_args = SFTConfig(
        output_dir=output_dir,
        max_seq_length=728,
        num_train_epochs=num_epochs,
        save_strategy="steps",
        save_steps=10,
        logging_dir=f"{output_dir}/logs",
        logging_steps=10,
        save_total_limit=2,
        report_to="none",
        disable_tqdm=False,
        load_best_model_at_end=False,
        evaluation_strategy="no",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        logging_first_step=True,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        args=train_args,
        formatting_func=formatting_prompts_func,
        data_collator=data_collator,
    )

    return trainer



In [241]:
from transformers import Trainer
import torch.nn.functional as F

class SoftDistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("soft_label")  # shape: [batch, num_classes]
        outputs = model(**inputs)
        logits = outputs.logits[:, -1, :]  # get logits of final token

        # Apply softmax and KL divergence
        student_log_probs = F.log_softmax(logits, dim=-1)
        target_probs = labels  # already soft labels from dataset
        loss = F.kl_div(student_log_probs, target_probs, reduction="batchmean")

        return (loss, outputs) if return_outputs else loss


In [242]:
only_sent_xnli = [example for example in dataset if example["task"] in ["sentiment", "xnli"]][:600]


In [246]:
LABEL_MAP = {
    "sentiment": ["positive", "negative", "neutral"],
    "xnli": ["true", "false", "neither"]
}

def preprocess_soft_label(example):
    label_names = LABEL_MAP.get(example["task"], [])
    label_vector = [example["soft_label"].get(label, 0.0) for label in label_names]
    return {
        "text": f"{example['prompt'].strip()}",
        "soft_label": torch.tensor(label_vector, dtype=torch.float32)
    }


In [218]:
model_name = "lelapa/InkubaLM-0.4B"
model, tokenizer, _ = setup_model_and_tokenizer_for_distillation(model_name)
model = apply_lora_adapters(model)

trainable params: 524,288 || all params: 422,463,488 || trainable%: 0.1241


In [223]:
tokenized = tokenizer(formatting_prompts_func(dataset[0]), return_tensors="pt")
print(tokenizer.decode(tokenized["input_ids"][0]))


<s> Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.
@user ynxu fha da kanada kudi shikenan duk kayan nan zasu iya zama naka no
### Response: negative


In [224]:
import os
output_path = "/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/distillation"
os.makedirs(output_path, exist_ok=True)

print(os.path.exists(output_path))  # Should return True
print(output_path)

True
/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/distillation


In [238]:
only_sent_xnli = dataset[0:600]

In [240]:
only_sent_xnli.shap

AttributeError: 'dict' object has no attribute 'shape'

In [232]:
trainer = setup_distillation_trainer(model, dataset, tokenizer, output_dir= output_path)
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/900 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/trl/trainer/utils.py:145: UserWarning: Could not find response key `
### Response:` in the following instance: </s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s><s> Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.
@user  asha ruwa lafiya oga
### Response: positive</s>. This instance will be ignored in loss calculation. Note, if this happens often, consider increasing the `max_length`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/utils.py:145: UserWarning: Could not find

Step,Training Loss
1,0.000000
10,0.000000
20,0.000000


/usr/local/lib/python3.11/dist-packages/trl/trainer/utils.py:145: UserWarning: Could not find response key `
### Response:` in the following instance: <s> Premise: Somo ambalo linajadiliwa wakati wa mahojiano ya kuingia ni mmenyuko wa kaya kwa matangazo ya barua.
Claim: Mahojiano ya maingizo ni pamoja na kidogo vile ambavyo watu wanavyochukulia matangazo ya barua pepe.
Is the claim entailed by the premise? Answer:
### Response: true</s>. This instance will be ignored in loss calculation. Note, if this happens often, consider increasing the `max_length`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/utils.py:145: UserWarning: Could not find response key `
### Response:` in the following instance: </s></s></s></s></s></s></s></s></s></s></s></s></s></s><s> Your task is to do translation from English to Hausa. Your output must be the translation only. Don't output any explanation.
september 2014 - disclosure and barring service (applicant)
### Response: Septemba 201

KeyboardInterrupt: 

In [200]:
print(tokenizer.name_or_path)
print(tokenizer.is_fast)  # Should be True


lelapa/InkubaLM-0.4B
True


In [244]:
def load_lora_model(model_name="meta-llama/Llama-2-7b-chat-hf", use_4bit=True):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=use_4bit,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer


In [247]:
LABEL_MAP = {
    "sentiment": ["positive", "negative", "neutral"],
    "xnli": ["true", "false", "neither"]
}

def preprocess_soft_label(example):
    label_names = LABEL_MAP.get(example["task"], [])
    label_vector = [example["soft_label"].get(label, 0.0) for label in label_names]
    return {
        "text": f"{example['prompt'].strip()}",
        "soft_label": torch.tensor(label_vector, dtype=torch.float32)
    }


In [248]:
dataset = Dataset.from_list([ex for ex in only_sent_xnli if ex["task"] in LABEL_MAP])
dataset = dataset.map(preprocess_soft_label)


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [264]:
tokenized_dataset = dataset.map(tokenize_function)


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [278]:
cleaned_data = preprocess_soft_labels(all_tasks)
dataset = Dataset.from_list(cleaned_data)

In [285]:
sentiment_xnli_raw = all_tasks[0:600]

In [286]:
len(sentiment_xnli_raw)

600

In [280]:
print(tokenized_dataset[0].keys())
# Should include: input_ids, attention_mask, soft_label


dict_keys(['prompt', 'soft_label', 'input_ids', 'attention_mask'])


In [293]:
import os
import torch
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorWithPadding,
    Trainer
)
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# ---------------------------
# 1. Label map
# ---------------------------
LABEL_MAP = {
    "sentiment": ["positive", "negative", "neutral"],
    "xnli": ["true", "false", "neither"]
}

# ---------------------------
# 2. Convert soft_label dicts to vectors
# ---------------------------
def preprocess_soft_labels(examples):
    processed = []
    for ex in examples:
        task = ex["task"]
        label_order = LABEL_MAP.get(task, [])
        if not label_order:
            continue

        soft_dict = ex.get("soft_label", {})
        label_vector = [float(soft_dict.get(label, 0.0) or 0.0) for label in label_order]

        if sum(label_vector) == 0:
            continue  # skip examples with no soft label info

        processed.append({
            "prompt": ex["prompt"].strip(),
            "soft_label": label_vector
        })
    return processed

# ---------------------------
# 3. Tokenize prompts
# ---------------------------
def tokenize_function(examples):
    encodings = tokenizer(
        examples["prompt"],
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    return {
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "soft_label": torch.tensor(examples["soft_label"], dtype=torch.float32)
    }


# ---------------------------
# 4. Load Lelapa InkubaLM-0.4B with LoRA + 4-bit
# ---------------------------
def load_lora_model(model_name="lelapa/InkubaLM-0.4B"):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer

# ---------------------------
# 5. Custom Trainer with KL loss
# ---------------------------
class SoftDistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        soft_labels = inputs.pop("soft_label")
        outputs = model(**inputs)
        logits = outputs.logits[:, -1, :]  # final token logits

        # Ensure soft_labels is a tensor (it should be now!)
        if not isinstance(soft_labels, torch.Tensor):
            soft_labels = torch.tensor(soft_labels, dtype=torch.float32)

        student_log_probs = F.log_softmax(logits, dim=-1)
        loss = F.kl_div(student_log_probs, soft_labels.to(student_log_probs.device), reduction="batchmean")
        return (loss, outputs) if return_outputs else loss

# ---------------------------
# 6. Main Training Function
# ---------------------------
def train_soft_distilled_lora_model(raw_data, model_name="lelapa/InkubaLM-0.4B"):
    print("Preprocessing data...")
    processed = preprocess_soft_labels(raw_data)
    dataset = Dataset.from_list(processed)

    print("Loading model and tokenizer...")
    model, tokenizer = load_lora_model(model_name)

    print("Tokenizing dataset...")
    tokenized_dataset = dataset.map(tokenize_function, remove_columns=dataset.column_names)
    print(tokenized_dataset[0])
    print(type(tokenized_dataset[0]["soft_label"]))  # should be <class 'torch.Tensor'>



    print("Starting training...")
    training_args = TrainingArguments(
        output_dir="./soft_distill_inkuba_output",
        per_device_train_batch_size=4,
        num_train_epochs=3,
        gradient_accumulation_steps=2,
        logging_steps=10,
        save_steps=50,
        logging_dir="./logs",
        report_to="none",
        eval_strategy="no",
        label_names=[],
    )

    trainer = SoftDistillationTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )

    trainer.train()

# ---------------------------
# 7. Run this with your data
# ---------------------------
if __name__ == "__main__":
    # 👇 Replace this with your full dataset

    train_soft_distilled_lora_model(sentiment_xnli_raw, model_name="lelapa/InkubaLM-0.4B")


Preprocessing data...
Loading model and tokenizer...
trainable params: 524,288 || all params: 422,463,488 || trainable%: 0.1241
Tokenizing dataset...


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

{'soft_label': [0.0, 0.9984400272369385, 5.999999848427251e-05], 'input_ids': [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

KeyError: 'soft_label'